In [1]:
!pip install anthropic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 17.3 MB/s eta 0:00:00


In [3]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

cevap = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[
        {"role": "user", "content": "phyton da liste nedir 1 cümle ile anlat."}
    ],
)

print(cevap.content[0].text)

# Phyton

Bitkilerde meristematik dokudan çıkan, bir yaprak ve ona karşılık gelen gövde kısmından oluşan tekrarlanan temel yapısal birimdir.


In [5]:
mesajlar = []

while True:
    soru = input("Sen: ")
    if soru == "çık":
        break

    mesajlar.append({"role": "user", "content": soru})

    cevap = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=mesajlar,
    )

    metin = cevap.content[0].text
    print("Claude:", metin)

    mesajlar.append({"role": "assistant", "content": metin})

Sen: Benim adım Fatih Mert
Claude: Merhaba Fatih Mert! 👋

Tanıştığımız için memnun oldum. Sana nasıl yardımcı olabilirim?
Sen: Adım neydi?
Claude: Senin adın **Fatih Mert**! 😊

Az önce kendini tanıtmıştın.
Sen: çık


In [9]:
import pandas as pd
import random

random.seed(1)
urunler = ["Kalem", "Defter", "Silgi", "Çanta", "Ajanda"]
fiyatlar = {"Kalem": 5, "Defter": 20, "Silgi": 3, "Çanta": 250, "Ajanda": 60}
aylar = ["Ocak", "Şubat", "Mart", "Nisan", "Mayıs", "Haziran"]

satirlar = []
for ay in aylar:
    for urun in urunler:
        adet = random.randint(5, 100)
        satirlar.append({"ay": ay, "urun": urun, "adet": adet, "fiyat": fiyatlar[urun]})

df = pd.DataFrame(satirlar)
df.to_csv("satislar.csv", index=False)
print(df.head())

     ay    urun  adet  fiyat
0  Ocak   Kalem    22      5
1  Ocak  Defter    77     20
2  Ocak   Silgi    13      3
3  Ocak   Çanta    37    250
4  Ocak  Ajanda    20     60


In [10]:
df = pd.read_csv("satislar.csv")
df["ciro"] = df["adet"] * df["fiyat"]
print(df.groupby("urun")["ciro"].sum())

urun
Ajanda    17700
Defter     6860
Kalem      1325
Silgi       573
Çanta     78750
Name: ciro, dtype: int64


In [12]:
def urun_cirosu():
    df = pd.read_csv("satislar.csv")
    df["ciro"] = df["adet"] * df["fiyat"]
    return df.groupby("urun")["ciro"].sum().to_string()

def aylik_ciro():
    df = pd.read_csv("satislar.csv")
    df["ciro"] = df["adet"] * df["fiyat"]
    return df.groupby("ay", sort=False)["ciro"].sum().to_string()

araclar = [
    {
        "name": "urun_cirosu",
        "description": "Her ürünün 6 aylık toplam cirosunu verir.",
        "input_schema": {"type": "object", "properties": {}},
    },
    {
        "name": "aylik_ciro",
        "description": "Her ayın toplam cirosunu verir.",
        "input_schema": {"type": "object", "properties": {}},
    },
]

fonksiyonlar = {"urun_cirosu": urun_cirosu, "aylik_ciro": aylik_ciro}

In [15]:
def agent(soru):
    mesajlar = [{"role": "user", "content": soru}]

    for tur in range(10):
        cevap = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=500,
            tools=araclar,
            messages=mesajlar,
        )

        if cevap.stop_reason != "tool_use":
            return "".join(b.text for b in cevap.content if b.type == "text")

        mesajlar.append({"role": "assistant", "content": cevap.content})

        sonuclar = []
        for blok in cevap.content:
            if blok.type == "tool_use":
                print("Araç çağrıldı:", blok.name)
                sonuc = fonksiyonlar[blok.name]()
                sonuclar.append({
                    "type": "tool_result",
                    "tool_use_id": blok.id,
                    "content": sonuc,
                })

        mesajlar.append({"role": "user", "content": sonuclar})

    return "Tur sınırına ulaşıldı"

print(agent("Ciro en düşük ürün ve en iyi ay hangisi?"))

Araç çağrıldı: urun_cirosu
Araç çağrıldı: aylik_ciro
Sonuçlara göre:

**En düşük ciro yapan ürün:** **Silgi** (573 TL)

**En iyi ay:** **Nisan** (29.175 TL)
